# LightGBM Importance

Same pattern as RF/XGBoost Importance. Capacity is matched to the
XGBoost config on purpose: `num_leaves=7` is the max leaf count a
depth-3 tree can have, so a difference in results reflects the
leaf-wise vs. level-wise growth strategy, not "one model being bigger
than the other".

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Locate the repo root without importing from src yet.
_current = Path.cwd().resolve()
PROJECT_ROOT = next(
    candidate
    for candidate in (_current, *_current.parents)
    if (candidate / "AGENTS.md").exists()
)
sys.path.insert(0, str(PROJECT_ROOT))

from src.feature_selection.data_loading import load_split, baseline_mean_metrics


In [2]:
X_train, y_train = load_split("train", processed_dir=PROJECT_ROOT / "data" / "processed")
X_val, y_val = load_split("validation", processed_dir=PROJECT_ROOT / "data" / "processed")

feature_cols = list(X_train.columns)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)


Train: (1895, 25)
Validation: (600, 25)


## 1. Fit once on train, rank all 25 features by gain-based importance

In [3]:
LGBM_PARAMS = dict(
    n_estimators=300,
    max_depth=3,
    num_leaves=7,          # 2^max_depth - 1, capacity-matched to XGBoost
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=-1,
)

lgbm_model = LGBMRegressor(**LGBM_PARAMS)
lgbm_model.fit(X_train, y_train)

importance_df = (
    pd.DataFrame({
        "feature": feature_cols,
        "importance": lgbm_model.feature_importances_,
    })
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

importance_df


,feature,importance
0,volume_sma_20,141
1,sma_60,141
2,volatility_20,126
3,macd_signal,118
4,sma_5,112
5,atr_14,86
6,price_to_sma_60,82
7,macd_hist,73
8,macd,64
9,volatility_5,63


## 2. Top-K subsets vs. the train-mean baseline

In [4]:
top_k_list = [5, 10, 15, 20]

results = [baseline_mean_metrics(y_train, y_val)]

for k in top_k_list:
    top_features = importance_df["feature"].head(k).tolist()

    model = LGBMRegressor(**LGBM_PARAMS)
    model.fit(X_train[top_features], y_train)

    y_pred = model.predict(X_val[top_features])

    mse = mean_squared_error(y_val, y_pred)

    results.append({
        "method": "LightGBM Importance",
        "n_selected_features": k,
        "selected_features": top_features,
        "RMSE": mse ** 0.5,
        "MAE": mean_absolute_error(y_val, y_pred),
        "R2": r2_score(y_val, y_pred),
    })

lgbm_results_df = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)
lgbm_results_df


,method,n_selected_features,selected_features,RMSE,MAE,R2
0,Baseline (predict train mean),0,[],0.105132,0.077442,-0.008612
1,LightGBM Importance,20,"[volume_sma_20, sma_60, volatility_20, macd_si...",0.109890,0.083027,-0.101962
2,LightGBM Importance,10,"[volume_sma_20, sma_60, volatility_20, macd_si...",0.112726,0.084842,-0.159577
3,LightGBM Importance,5,"[volume_sma_20, sma_60, volatility_20, macd_si...",0.113230,0.087985,-0.169963
4,LightGBM Importance,15,"[volume_sma_20, sma_60, volatility_20, macd_si...",0.114242,0.086425,-0.190984


## 3. Cross-check against RandomForest / XGBoost top features

In [5]:
results_dir = PROJECT_ROOT / "data" / "processed" / "embedded_results"

rf_path = results_dir / "random_forest_importance_full.csv"
xgb_path = results_dir / "xgboost_importance_full.csv"

lgbm_top10 = set(importance_df["feature"].head(10))

if rf_path.exists() and xgb_path.exists():
    rf_top10 = set(pd.read_csv(rf_path)["feature"].head(10))
    xgb_top10 = set(pd.read_csv(xgb_path)["feature"].head(10))

    all_three = rf_top10 & xgb_top10 & lgbm_top10
    print("All 3 models agree (top10):", all_three)
    print("LightGBM only:", lgbm_top10 - rf_top10 - xgb_top10)
else:
    print("Run random_forest_importance.ipynb and xgboost_importance.ipynb first.")


All 3 models agree (top10): {'sma_5', 'atr_14', 'macd_hist', 'sma_60', 'volatility_20'}
LightGBM only: {'volatility_5'}


In [6]:
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "embedded_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

importance_df.to_csv(OUTPUT_DIR / "lightgbm_importance_full.csv", index=False)
lgbm_results_df.to_csv(OUTPUT_DIR / "lightgbm_importance_results.csv", index=False)

print("Saved to:", OUTPUT_DIR)


Saved to: /Users/yangjaehoon/Desktop/StockLens/data/processed/embedded_results


## 4. Early stopping retry

Same idea as the XGBoost retry: raise `n_estimators` to a high ceiling
and let validation-monitored early stopping decide when to actually
stop, instead of forcing exactly 300 rounds regardless of overfitting.
Same caveat about validation being used to pick the stopping point
applies here too.

In [7]:
import lightgbm

LGBM_ES_PARAMS = dict(
    n_estimators=1000,
    max_depth=3,
    num_leaves=7,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=-1,
)

results_es = [baseline_mean_metrics(y_train, y_val)]

for k in top_k_list:
    top_features = importance_df["feature"].head(k).tolist()

    model = LGBMRegressor(**LGBM_ES_PARAMS)
    model.fit(
        X_train[top_features], y_train,
        eval_set=[(X_val[top_features], y_val)],
        eval_metric="rmse",
        callbacks=[lightgbm.early_stopping(stopping_rounds=30, verbose=False)],
    )

    y_pred = model.predict(X_val[top_features])
    mse = mean_squared_error(y_val, y_pred)

    results_es.append({
        "method": "LightGBM Importance (early stopping)",
        "n_selected_features": k,
        "selected_features": top_features,
        "best_iteration": model.best_iteration_,
        "RMSE": mse ** 0.5,
        "MAE": mean_absolute_error(y_val, y_pred),
        "R2": r2_score(y_val, y_pred),
    })

lgbm_es_results_df = pd.DataFrame(results_es).sort_values("RMSE").reset_index(drop=True)
lgbm_es_results_df


/Users/yangjaehoon/Desktop/StockLens/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/Users/yangjaehoon/Desktop/StockLens/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/Users/yangjaehoon/Desktop/StockLens/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/Users/yangjaehoon/Desktop/StockLens/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated,

,method,n_selected_features,selected_features,RMSE,MAE,R2,best_iteration
0,LightGBM Importance (early stopping),20,"[volume_sma_20, sma_60, volatility_20, macd_si...",0.102448,0.075882,0.042240,28.0
1,LightGBM Importance (early stopping),15,"[volume_sma_20, sma_60, volatility_20, macd_si...",0.102996,0.076091,0.031967,20.0
2,LightGBM Importance (early stopping),10,"[volume_sma_20, sma_60, volatility_20, macd_si...",0.103065,0.076180,0.030673,10.0
3,LightGBM Importance (early stopping),5,"[volume_sma_20, sma_60, volatility_20, macd_si...",0.103232,0.077175,0.027514,9.0
4,Baseline (predict train mean),0,[],0.105132,0.077442,-0.008612,NaN


In [8]:
lgbm_es_results_df.to_csv(
    PROJECT_ROOT / "data" / "processed" / "embedded_results" / "lightgbm_importance_early_stopping_results.csv",
    index=False,
)
print("Saved.")


Saved.
